### Human in the Loop (HITL)

source: langgraph Documentation 

The Human-in-the-Loop (HITL) middleware lets you add human oversight to agent tool calls. When a model proposes an action that might require review—for example, writing to a file or executing SQL—the middleware can pause execution and wait for a decision.

It does this by checking each tool call against a configurable policy. If intervention is needed, the middleware issues an interrupt that halts execution. 

The graph state is saved using LangGraph’s persistence layer, so execution can pause safely and resume later.

A human decision then determines what happens next: the action can be approved as-is (approve), modified before running (edit), rejected with feedback (reject), or responded to directly (respond) for “ask user” style tools.


#### Interrupt Decision Types
The middleware defines four built-in ways a human can respond to an interrupt:


| Decision Type | Description | Example Use Case |
|:---:|---|---|
| ✅ **approve** | Execute the tool with the original arguments as proposed by the agent. | Send an email draft exactly as written |
| ✏️ **edit** | Modify the tool arguments before execution. | Change the recipient before sending an email |
| ❌ **reject** | Skip executing this tool call entirely and return rejection feedback to the agent. | Deny file deletion and explain why |
| 💬 **respond** | Return the human's message directly as a synthetic tool result, skipping execution, for "ask user" style tools. | Answer an "ask_user" prompt with a direct reply |



The available decision types for each tool depend on the policy you configure in interrupt_on. When multiple tool calls are paused at the same time, each action requires a separate decision. Decisions must be provided in the same order as the actions appear in the interrupt request.

Use reject when the human is denying the requested action. Use respond only when the human is acting as the tool, such as answering an ask_user prompt. Do not use respond to deny side-effecting tools, because its message is treated as a successful tool result.

#### Why Human in the Loop Exists

LLM agents can hallucinate, misinterpret intent, or take irreversible actions (deleting records, sending emails, charging credit cards). Human-in-the-loop (HITL) is the mechanism that lets you pause a running graph, surface its current state to a human, and then either approve, reject, or modify that state before execution continues.

LangGraph implements HITL natively through:
1. Interrupts — pause execution at a node boundary
2. Resume — continue execution from where it paused
3. update_state() — inject human-modified state before resuming
4. Approval Workflows — structured yes/no gates before consequential actions
5. Manual Review — surfacing intermediate artifacts (drafts, plans) for human editing
6. Feedback Loops — collecting structured human feedback and routing based on it


All of these depend on checkpointing. without a checkpointer , there is no pause/resume-- The graph has no memory of where it stopped.

#### Core Mental Model
``` markdown
Graph running →  reaches interrupt node  →  PAUSED (state saved to checkpointer)
                                                     ↓
                                          Human reviews state
                                                     ↓
                          Human approves / edits state / rejects
                                                     ↓
              graph.invoke(None, config)  →  RESUMED from checkpoint
```

The key insight: when a graph is interrupted, you do NOT re-invoke with the original input. You re-invoke with None as the input and the same thread_id in the config. LangGraph reads the saved checkpoint and picks up exactly where it left off.


In [8]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()
api_key = os.getenv("groq_api_key")
model_name = os.getenv("groq_model_name")

from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict, Annotated
import operator

llm = ChatGroq(
    api_key = os.getenv("groq_api_key"),
    model_name = model_name
)
print(llm)

metadata={'versions': {'langchain-core': '1.4.6', 'langchain': '1.3.8'}} output_version=None profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True} client=<groq.resources.chat.completions.Completions object at 0x0000021BA2296A10> async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000021BA233D210> model_name='llama-3.3-70b-versatile' model_kwargs={} groq_api_key=SecretStr('**********') groq_api_base=None groq_proxy=None


### Part 1 — Interrupt
1. What is interrupt_before?

```interrupt_before``` is a compile-time option that tells LangGraph: "before executing this node, pause and wait for human input." The node itself is NOT called until the human resumes.

``` python
graph = builder.compile(
    checkpointer=MemorySaver(),
    interrupt_before=["node_name"]   # list of node names to pause before
)
```

we can also use interrupt_after to pause AFTER a node runs (useful when you want to inspect what the node produced before continuing).

``` python
graph = builder.compile(
    checkpointer=MemorySaver(),
    interrupt_after=["node_name"]    # pause after node runs, before next node
)
```







In [ ]:
### working example

from langgraph.graph import StateGraph, START,END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict
from langgraph.types import interrupt, Command

class State(TypedDict):
    task: str
    plan: str
    approved: bool
    result: str

def plan_node(state: State) -> dict:
    """LLM generates a plan for the task."""
    plan = f"Plan for '{state['task']}': Step 1 → Step 2 → Step 3"
    print(f"[plan_node] Generated plan: {plan}")
    return {"plan": plan}

def execute_node(state: State) -> dict:
    """Executes the approved plan."""
    print(f"[execute_node] Executing: {state['plan']}")
    return {"result": f"Completed: {state['plan']}"}

builder = StateGraph(State)
builder.add_node("plan", plan_node)
builder.add_node("execute", execute_node)
builder.set_entry_point("plan")
builder.add_edge("plan", "execute")
builder.add_edge("execute", END)

checkpointer = MemorySaver()

graph = builder.compile(
    checkpointer=checkpointer,
    interrupt_before=["execute"]   # ← pause BEFORE execute runs
)
# --- Run 1: Graph pauses before execute ---
config = {"configurable": {"thread_id": "thread-001"}}
initial_state = {"task": "Send marketing email to all users", "approved": False}
result = graph.invoke(initial_state, config)
print("\n--- Graph paused. Current state: ---")
print(result)

# --- Human reviews the plan here ---
print("\nHuman reviewing plan...")
print(f"Plan: {result['plan']}")

# --- Run 2: Resume execution ---
print("\nHuman approved. Resuming...")
final = graph.invoke(None, config)   # ← None input, same thread_id
print("\n--- Final state: ---")
print(final)




[plan_node] Generated plan: Plan for 'Send marketing email to all users': Step 1 → Step 2 → Step 3

--- Graph paused. Current state: ---
{'task': 'Send marketing email to all users', 'plan': "Plan for 'Send marketing email to all users': Step 1 → Step 2 → Step 3", 'approved': False}

Human reviewing plan...
Plan: Plan for 'Send marketing email to all users': Step 1 → Step 2 → Step 3

Human approved. Resuming...
[execute_node] Executing: Plan for 'Send marketing email to all users': Step 1 → Step 2 → Step 3

--- Final state: ---
{'task': 'Send marketing email to all users', 'plan': "Plan for 'Send marketing email to all users': Step 1 → Step 2 → Step 3", 'approved': False, 'result': "Completed: Plan for 'Send marketing email to all users': Step 1 → Step 2 → Step 3"}


In [13]:
# new practical implementation

class CodeState(TypedDict):
    user_request: str
    generated_code: str
    human_feedback: str

# node 1
def generate_code(state: CodeState):

    prompt = f"""
    Write Python code.

    Requirement:
    {state["user_request"]}
    """

    response = llm.invoke(prompt)

    return {
        "generated_code": response.content
    }

# node 2. Human review (interrupt)
def human_review(state: CodeState):

    feedback = interrupt(
        {
            "generated_code": state["generated_code"],
            "message": "Approve this code or provide modifications."
        }
    )

    return {
        "human_feedback": feedback
    }

# decision function

def review_decision(state: CodeState):

    feedback = state["human_feedback"].lower()

    if feedback == "approve":
        return "approved"

    return "modify"

# node 3 : modufy the code

def modify_code(state: CodeState):
    prompt = f"""
    Here is the existing code:

    {state["generated_code"]}

    Human feedback:

    {state["human_feedback"]}

    Modify the code accordingly.
    """

    response = llm.invoke(prompt)

    return {
        "generated_code": response.content
    }

builder = StateGraph(CodeState)

builder.add_node("generate", generate_code)
builder.add_node("review", human_review)
builder.add_node("modify", modify_code)

builder.add_edge(START, "generate")
builder.add_edge("generate", "review")

builder.add_conditional_edges(
    "review",
    review_decision,
    {
        "approved": END,
        "modify": "modify"
    }
)

builder.add_edge("modify", "review")

# memory

memory = MemorySaver()

graph = builder.compile(
    checkpointer=memory
)


In [14]:
config = {
    "configurable": {
        "thread_id": "thread-1"
    }
}

result = graph.invoke(
    {
        "user_request": "Write a Python function to sort a list."
    },
    config=config
)

print(result)

{'user_request': 'Write a Python function to sort a list.', 'generated_code': '**Sorting a List in Python**\n================================\n\nHere\'s a simple Python function that uses the built-in `sorted()` function to sort a list.\n\n```python\ndef sort_list(input_list):\n    """\n    Sorts a list in ascending order.\n\n    Args:\n        input_list (list): The list to be sorted.\n\n    Returns:\n        list: The sorted list.\n    """\n    return sorted(input_list)\n\n# Example usage:\nnumbers = [64, 34, 25, 12, 22, 11, 90]\nprint("Original list:", numbers)\nprint("Sorted list:", sort_list(numbers))\n```\n\n**Output:**\n```\nOriginal list: [64, 34, 25, 12, 22, 11, 90]\nSorted list: [11, 12, 22, 25, 34, 64, 90]\n```\n\nIf you want to implement a sorting algorithm from scratch, here\'s an example using the QuickSort algorithm:\n\n```python\ndef quicksort(arr):\n    """\n    Sorts a list using the QuickSort algorithm.\n\n    Args:\n        arr (list): The list to be sorted.\n\n    

In [15]:
graph.invoke(
    Command(
        resume="Use merge sort instead of bubble sort."
    ),
    config=config
)

{'user_request': 'Write a Python function to sort a list.',
 'generated_code': '## Sorting a List in Python\n================================\n\nHere\'s a simple Python function that uses the built-in `sorted()` function to sort a list.\n\n```python\ndef sort_list(input_list):\n    """\n    Sorts a list in ascending order.\n\n    Args:\n        input_list (list): The list to be sorted.\n\n    Returns:\n        list: The sorted list.\n    """\n    return sorted(input_list)\n\n# Example usage:\nnumbers = [64, 34, 25, 12, 22, 11, 90]\nprint("Original list:", numbers)\nprint("Sorted list:", sort_list(numbers))\n```\n\n## Output:\n```\nOriginal list: [64, 34, 25, 12, 22, 11, 90]\nSorted list: [11, 12, 22, 25, 34, 64, 90]\n```\n\nIf you want to implement a sorting algorithm from scratch, here\'s an example using the Merge Sort algorithm:\n\n```python\ndef merge_sort(arr):\n    """\n    Sorts a list using the Merge Sort algorithm.\n\n    Args:\n        arr (list): The list to be sorted.\n\n  

In [16]:
graph.invoke(
    Command(
        resume="Add comments and type hints."
    ),
    config=config
)

{'user_request': 'Write a Python function to sort a list.',
 'generated_code': '### Modified Code\n\nHere is the modified code with added comments and type hints:\n\n```python\ndef sort_list(input_list: list) -> list:\n    """\n    Sorts a list in ascending order using the built-in sorted() function.\n\n    Args:\n        input_list (list): The list to be sorted.\n\n    Returns:\n        list: The sorted list.\n    """\n    # Use the built-in sorted() function to sort the list\n    return sorted(input_list)\n\n# Example usage:\nnumbers = [64, 34, 25, 12, 22, 11, 90]\nprint("Original list:", numbers)\nprint("Sorted list:", sort_list(numbers))\n\n\ndef merge_sort(arr: list) -> list:\n    """\n    Sorts a list using the Merge Sort algorithm.\n\n    Args:\n        arr (list): The list to be sorted.\n\n    Returns:\n        list: The sorted list.\n    """\n    # Base case: If the list has one or zero elements, it is already sorted\n    if len(arr) <= 1:\n        return arr\n\n    # Divide t

In [17]:
graph.invoke(
    Command(
        resume="approve"
    ),
    config=config
)

{'user_request': 'Write a Python function to sort a list.',
 'generated_code': '### Modified Code\n\nHere is the modified code with added comments and type hints:\n\n```python\ndef sort_list(input_list: list) -> list:\n    """\n    Sorts a list in ascending order using the built-in sorted() function.\n\n    Args:\n        input_list (list): The list to be sorted.\n\n    Returns:\n        list: The sorted list.\n    """\n    # Use the built-in sorted() function to sort the list\n    return sorted(input_list)\n\n# Example usage:\nnumbers = [64, 34, 25, 12, 22, 11, 90]\nprint("Original list:", numbers)\nprint("Sorted list:", sort_list(numbers))\n\n\ndef merge_sort(arr: list) -> list:\n    """\n    Sorts a list using the Merge Sort algorithm.\n\n    Args:\n        arr (list): The list to be sorted.\n\n    Returns:\n        list: The sorted list.\n    """\n    # Base case: If the list has one or zero elements, it is already sorted\n    if len(arr) <= 1:\n        return arr\n\n    # Divide t

#### What happens step by step

* ```graph.invoke(initial_state, config)``` — plan_node runs, state is saved, graph pauses before execute

* You inspect result — the plan is visible

* graph.invoke(None, config) — LangGraph reads the checkpoint, skips plan_node (already done), runs execute_node

#### 1.3 interrupt_after Example

Use interrupt_after when the node should run first, and you want to inspect or modify what it produced.

In [20]:
graph = builder.compile(
    checkpointer=MemorySaver(),
    interrupt_after=["plan"]   # plan runs, THEN pause
)

config = {"configurable": {"thread_id": "thread-002"}}
result = graph.invoke({"task": "Deploy to production"}, config)

# plan_node has already run — we can see the plan
print(f"Plan generated: {result['plan']}")

# Human can modify the plan via update_state (covered in Part 3)
# Then resume
final = graph.invoke(None, config)

[plan_node] Generated plan: Plan for 'Deploy to production': Step 1 → Step 2 → Step 3
Plan generated: Plan for 'Deploy to production': Step 1 → Step 2 → Step 3
[execute_node] Executing: Plan for 'Deploy to production': Step 1 → Step 2 → Step 3


In [21]:
# 1.4 Checking Graph State After Interrupt

# Use get_state() to inspect what the graph knows at the paused checkpoint.

snapshot = graph.get_state(config)

print(snapshot.values)          # the full state dict
print(snapshot.next)            # which node(s) will run on resume
print(snapshot.metadata)        # step count, source info

{'task': 'Deploy to production', 'plan': "Plan for 'Deploy to production': Step 1 → Step 2 → Step 3", 'result': "Completed: Plan for 'Deploy to production': Step 1 → Step 2 → Step 3"}
()
{'source': 'loop', 'step': 2, 'parents': {}}


```snapshot.next``` is your clearest signal that the graph is paused — it contains the name(s) of the node(s) waiting to run.



In [22]:
# Pattern to check if graph is paused
snapshot = graph.get_state(config)
if snapshot.next:
    print(f"Graph is paused. Waiting on: {snapshot.next}")
else:
    print("Graph has completed.")

Graph has completed.


### Resume

interrupt pauses. Resume Restart from the paused state.

1. The Resume Mechanic

Resume is simply re-invoking the graph with None as input and the same thread config.

``` python
# Pause
graph.invoke(initial_input, config)

# Resume
graph.invoke(None, config)
```
LangGraph does the rest — reads the checkpoint, knows which node to run next, and continues.

2. Resuming Multiple Times

A graph can be interrupted and resumed multiple times across its lifetime — once per interrupt_before node in the execution path.




In [23]:
## Example
class PipelineState(TypedDict):
    data: str
    cleaned_data: str
    analysis: str
    report: str

def clean_node(state): 
    return {"cleaned_data": f"cleaned({state['data']})"}

def analyze_node(state): 
    return {"analysis": f"analysis of {state['cleaned_data']}"}

def report_node(state): 
    return {"report": f"Report: {state['analysis']}"}

builder = StateGraph(PipelineState)
builder.add_node("clean", clean_node)
builder.add_node("analyze", analyze_node)
builder.add_node("report", report_node)
builder.set_entry_point("clean")
builder.add_edge("clean", "analyze")
builder.add_edge("analyze", "report")
builder.add_edge("report", END)

graph = builder.compile(
    checkpointer=MemorySaver(),
    interrupt_before=["analyze", "report"]  # two interrupts
)

config = {"configurable": {"thread_id": "pipeline-001"}}

# Run 1: clean runs, pauses before analyze
s1 = graph.invoke({"data": "raw_data"}, config)
print(f"Paused 1. Next: {graph.get_state(config).next}")  # ('analyze',)

# Run 2: analyze runs, pauses before report
s2 = graph.invoke(None, config)
print(f"Paused 2. Next: {graph.get_state(config).next}")  # ('report',)

# Run 3: report runs, graph completes
s3 = graph.invoke(None, config)
print(f"Done. Next: {graph.get_state(config).next}")      # ()
print(s3)

Paused 1. Next: ('analyze',)
Paused 2. Next: ('report',)
Done. Next: ()
{'data': 'raw_data', 'cleaned_data': 'cleaned(raw_data)', 'analysis': 'analysis of cleaned(raw_data)', 'report': 'Report: analysis of cleaned(raw_data)'}


3. Resuming with Stream

You can also resume using stream() for real-time visibility into what each node produces as it runs.

``` python
# Resume with streaming output
for chunk in graph.stream(None, config, stream_mode="values"):
    print(chunk)
```




#### Part 3 — update_state()

```update_state()``` let you directly modified the graph's stored state between an interrupt and resume. This how we can inject a human edits. corrections to a draft, changes to a plan, setting an approval flag, etc.

``` python
graph.update_state(
    config,           # same config / thread_id
    {"key": "new_value"},   # the state updates to apply
    as_node="node_name"     # optional: pretend this update came from this node
)
```

After update_state, the next graph.invoke(None, config) will see the modified state.



In [11]:
# Full Example: Human Edits a Draft.
from langgraph.graph import StateGraph, END, START
from langgraph.graph.message import add_messages
from typing import TypedDict, Annotated


class WritingState(TypedDict):
    topic: str
    draft: str
    final: str

def draft_node(state: WritingState) -> dict:
    draft = f"[AI Draft] An article about {state['topic']}. This needs editing."
    return {"draft": draft}

def publish_node(state: WritingState) -> dict:
    return {"final": f"[PUBLISHED] {state['draft']}"}

builder = StateGraph(WritingState)
builder.add_node("draft", draft_node)
builder.add_node("publish", publish_node)
builder.set_entry_point("draft")
builder.add_edge("draft", "publish")
builder.add_edge("publish", END)

graph = builder.compile(
    checkpointer=MemorySaver(),
    interrupt_after=["draft"]  # pause after draft is created
)

config = {"configurable": {"thread_id": "writing-001"}}
result = graph.invoke({"topic": "LangGraph HITL"}, config)

print(f"\nAI Draft:\n{result['draft']}")
# "[AI Draft] An article about LangGraph HITL. This needs editing."

# Human edits the draft
human_edited = "Human-edited: LangGraph's Human-in-the-Loop is powerful because..."

# Inject the edit into state
graph.update_state(
    config,
    {"draft": human_edited}
)

# Verify the state was updated
snapshot = graph.get_state(config)
print(f"\nUpdated state draft:\n{snapshot.values['draft']}")

# Resume — publish_node will now use the human-edited draft
final = graph.invoke(None, config)
print(f"\nFinal published:\n{final['final']}")

TypeError: Reviver.__init__() got an unexpected keyword argument 'allowed_objects'

In [7]:
from importlib.metadata import version

print("langgraph:", version("langgraph"))
print("langchain-core:", version("langchain-core"))
print("langchain:", version("langchain"))


langgraph: 1.2.9
langchain-core: 1.4.9
langchain: 1.3.13


In [8]:
from importlib.metadata import version
print("jsonpatch:", version("jsonpatch"))
print("jsonpointer:", version("jsonpointer"))


jsonpatch: 1.33
jsonpointer: 3.1.1


In [9]:
!pip install --upgrade jsonpatch jsonpointer


In [10]:
!pip show langgraph


Name: langgraph
Version: 1.2.9
Summary: Building stateful, multi-actor applications with LLMs
Home-page: https://docs.langchain.com/oss/python/langgraph/overview
Author: 
Author-email: 
License-Expression: MIT
Location: c:\users\subramani.v\appdata\local\programs\python\python310\lib\site-packages
Requires: langchain-core, langgraph-checkpoint, langgraph-prebuilt, langgraph-sdk, pydantic, xxhash
Required-by: langchain, langchain-google-community


In [12]:
print('subramani')

subramani


#### The as_node Parameter

as_node tells LangGraph which node the update is "coming from." This matters because LangGraph uses the source node to determine what to run next.

``` python 
# Update state pretending the "draft" node produced this output
# LangGraph will then proceed to the node that follows "draft"
graph.update_state(
    config,
    {"draft": "Human-written draft"},
    as_node="draft"
)
```

Without as_node, LangGraph applies the update to the most recent checkpoint. With it, you can control the flow more explicitly.

#### Part 4 — Approval Workflows



##### 4.1 Pattern: Explicit Approval Gate

The Most common HITL pattern. An Agent proposes an action. a human approves or rejects. only on the approval does the action execute.

In [ ]:

from typing import Literal,TypedDict

class ApprovalState(TypedDict):
    action: str
    params: dict
    human_decision: Literal["approved", "rejected", "pending"]
    outcome: str

def propose_action(state: ApprovalState) -> dict:
    """Agent proposes what it wants to do."""
    action = "DELETE all records older than 90 days"
    params = {"table": "user_sessions", "days": 90}
    print(f"[propose_action] Proposing: {action}")
    return {
        "action": action,
        "params": params,
        "human_decision": "pending"
    }

def execute_action(state: ApprovalState) -> dict:
    """Only runs if human approved."""
    print(f"[execute_action] Executing: {state['action']}")
    return {"outcome": f"Successfully executed: {state['action']}"}

def reject_action(state: ApprovalState) -> dict:
    """Handles rejection."""
    print(f"[reject_action] Action rejected by human.")
    return {"outcome": "Action was rejected. No changes made."}

def route_on_decision(state: ApprovalState) -> str:
    """Routes based on human_decision field."""
    return state["human_decision"]

builder = StateGraph(ApprovalState)
builder.add_node("propose", propose_action)
builder.add_node("execute", execute_action)
builder.add_node("reject", reject_action)
builder.set_entry_point("propose")

builder.add_conditional_edges(
    "propose",
    route_on_decision,
    {
        "approved": "execute",
        "rejected": "reject",
        "pending": END          # will be handled by interrupt
    }
)
builder.add_edge("execute", END)
builder.add_edge("reject", END)

graph = builder.compile(
    checkpointer=MemorySaver(),
    interrupt_after=["propose"]   # pause after proposal, before routing
)

config = {"configurable": {"thread_id": "approval-001"}}

# Step 1: Agent proposes the action
result = graph.invoke({"human_decision": "pending"}, config)
print(f"\nProposed action: {result['action']}")
print(f"Params: {result['params']}")

# Step 2: Human decides
user_input = input("\nApprove or Reject? (approved/rejected): ").strip()

# Step 3: Inject decision into state
graph.update_state(config, {"human_decision": user_input})

# Step 4: Resume — conditional edge now routes correctly
final = graph.invoke(None, config)
print(f"\nOutcome: {final['outcome']}")




NameError: name 'StateGraph' is not defined

#### 4.2 Pattern: Tool-Level Approval (Interrupt on Tool Call)

In tool-calling agents, you often want to approve each tool call before it executes. LangGraph's interrupt_before=["tools"] pattern handles this.


In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email to a recipient."""
    # In real code, this calls SendGrid / SES etc.
    return f"Email sent to {to} with subject '{subject}'"

@tool
def delete_record(record_id: str) -> str:
    """Delete a database record by ID."""
    return f"Record {record_id} deleted"

llm = ChatOpenAI(model="gpt-4o-mini")
tools = [send_email, delete_record]

checkpointer = MemorySaver()

# create_react_agent automatically creates a "tools" node
agent = create_react_agent(
    llm,
    tools,
    checkpointer=checkpointer,
    interrupt_before=["tools"]   # pause before ANY tool is called
)

config = {"configurable": {"thread_id": "agent-tool-approval"}}

# Run the agent — it will pause before calling any tool
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Send a welcome email to user@example.com"}]},
    config
)

# Inspect the pending tool call
snapshot = agent.get_state(config)
last_message = snapshot.values["messages"][-1]

print(f"\nAgent wants to call:")
for tool_call in last_message.tool_calls:
    print(f"  Tool: {tool_call['name']}")
    print(f"  Args: {tool_call['args']}")

# Human approves or rejects
decision = input("\nApprove tool call? (yes/no): ")

if decision.lower() == "yes":
    # Resume — tool will execute
    final = agent.invoke(None, config)
    print("\nTool executed. Final response:")
    print(final["messages"][-1].content)
else:
    # Reject by overriding with a ToolMessage saying it was denied
    from langchain_core.messages import ToolMessage
    tool_call_id = last_message.tool_calls[0]["id"]
    
    agent.update_state(
        config,
        {"messages": [ToolMessage(
            content="Tool call rejected by human reviewer.",
            tool_call_id=tool_call_id
        )]},
        as_node="tools"
    )
    final = agent.invoke(None, config)
    print("\nTool rejected. Agent response:")
    print(final["messages"][-1].content)

#### Part 5 — Manual Review

Manual review is about surfacing intermediate artifacts — not just yes/no decisions but actual content that humans read, edit, and return enriched.

In [ ]:
class DocumentState(TypedDict):
    user_request: str
    ai_draft: str
    reviewed_draft: str
    review_comments: str
    final_document: str
    status: str

def generate_draft(state: DocumentState) -> dict:
    """AI generates initial document."""
    draft = f"""
    CONTRACT DRAFT
    ==============
    Parties: Client and Vendor
    Subject: {state['user_request']}
    Terms: Standard 30-day payment terms apply.
    [AI-generated placeholder clauses here]
    """
    return {"ai_draft": draft, "status": "pending_review"}

def incorporate_review(state: DocumentState) -> dict:
    """Takes human-reviewed draft and finalizes."""
    final = f"""
    FINAL CONTRACT
    ==============
    {state['reviewed_draft']}
    
    Review Notes Applied: {state['review_comments']}
    Status: APPROVED FOR SIGNING
    """
    return {"final_document": final, "status": "complete"}

builder = StateGraph(DocumentState)
builder.add_node("generate", generate_draft)
builder.add_node("finalize", incorporate_review)
builder.set_entry_point("generate")
builder.add_edge("generate", "finalize")
builder.add_edge("finalize", END)

graph = builder.compile(
    checkpointer=MemorySaver(),
    interrupt_after=["generate"]
)

config = {"configurable": {"thread_id": "doc-review-001"}}

result = graph.invoke(
    {"user_request": "Software development services for 6 months"},
    config
)

# Surface the draft for review
print("\n" + "="*50)
print("DOCUMENT READY FOR MANUAL REVIEW")
print("="*50)
print(result["ai_draft"])
print("="*50)

# Simulate human review
human_reviewed = result["ai_draft"].replace(
    "[AI-generated placeholder clauses here]",
    "Clause 1: Payment within 30 days of invoice.\nClause 2: IP belongs to Client upon full payment."
)
review_comments = "Added specific payment and IP clauses. Removed vague placeholder."

# Inject reviewed content
graph.update_state(config, {
    "reviewed_draft": human_reviewed,
    "review_comments": review_comments
})

# Resume
final = graph.invoke(None, config)
print("\nFINAL DOCUMENT:")
print(final["final_document"])

In [ ]:
# 5.2 Pattern: Multi-Stage Review Pipeline
# For high-stakes content — medical, legal, financial — you may need multiple independent reviewers before an 
# artifact is approved.

from typing import List

class MultiReviewState(TypedDict):
    content: str
    reviews: Annotated[List[dict], operator.add]   # accumulates reviews
    final_approval: bool
    output: str

def ai_generate(state: MultiReviewState) -> dict:
    return {"content": "AI-generated medical protocol draft..."}

def legal_review_gate(state: MultiReviewState) -> dict:
    """Placeholder — interrupted before this runs for human review."""
    return {}

def medical_review_gate(state: MultiReviewState) -> dict:
    """Placeholder — second review gate."""
    return {}

def finalize(state: MultiReviewState) -> dict:
    all_approved = all(r["approved"] for r in state["reviews"])
    return {
        "final_approval": all_approved,
        "output": state["content"] if all_approved else "REJECTED — see review notes"
    }

builder = StateGraph(MultiReviewState)
builder.add_node("generate", ai_generate)
builder.add_node("legal_review", legal_review_gate)
builder.add_node("medical_review", medical_review_gate)
builder.add_node("finalize", finalize)
builder.set_entry_point("generate")
builder.add_edge("generate", "legal_review")
builder.add_edge("legal_review", "medical_review")
builder.add_edge("medical_review", "finalize")
builder.add_edge("finalize", END)

graph = builder.compile(
    checkpointer=MemorySaver(),
    interrupt_before=["legal_review", "medical_review"]
)

config = {"configurable": {"thread_id": "multi-review-001"}}

# Stage 1 — AI generates
graph.invoke({"reviews": []}, config)
print("Paused for legal review.")
print(f"Content: {graph.get_state(config).values['content']}")

# Legal reviewer adds their review via update_state
graph.update_state(config, {
    "reviews": [{"reviewer": "legal", "approved": True, "notes": "Legally sound."}]
})
graph.invoke(None, config)  # legal_review runs (no-op), pauses before medical_review

print("\nPaused for medical review.")

# Medical reviewer adds their review
graph.update_state(config, {
    "reviews": [{"reviewer": "medical", "approved": True, "notes": "Clinically appropriate."}]
})

final = graph.invoke(None, config)
print(f"\nFinal approval: {final['final_approval']}")
print(f"Output: {final['output']}")
print(f"All reviews: {final['reviews']}")


#### Part 6 — Feedback Loops

Feedback loops go beyond approval/rejection — they collect structured human feedback and route the agent to revise, retry, or escalate.

* 6.1 Pattern: Revision Loop
The agent produces output → human provides feedback → agent revises → loop until accepted.



In [ ]:
from typing import List

class RevisionState(TypedDict):
    request: str
    draft: str
    feedback_history: Annotated[List[dict], operator.add]
    iteration: int
    accepted: bool
    final: str

def write_draft(state: RevisionState) -> dict:
    iteration = state.get("iteration", 0) + 1
    
    if iteration == 1:
        draft = f"Initial draft for: {state['request']}. [Generic version]"
    else:
        # In real code the LLM would incorporate last feedback
        last_feedback = state["feedback_history"][-1]["comment"]
        draft = f"Revised draft (v{iteration}) incorporating: '{last_feedback}'"
    
    print(f"\n[write_draft] Iteration {iteration}: {draft}")
    return {"draft": draft, "iteration": iteration}

def apply_feedback(state: RevisionState) -> dict:
    """No-op placeholder — interrupted before this for human input."""
    return {}

def route_after_feedback(state: RevisionState) -> str:
    if state["accepted"]:
        return "finalize"
    elif state["iteration"] >= 3:
        return "escalate"
    else:
        return "revise"

def finalize_content(state: RevisionState) -> dict:
    print(f"[finalize] Accepted at iteration {state['iteration']}")
    return {"final": state["draft"]}

def escalate_content(state: RevisionState) -> dict:
    print("[escalate] Max iterations reached. Escalating to senior editor.")
    return {"final": "ESCALATED: Needs senior review"}

builder = StateGraph(RevisionState)
builder.add_node("write", write_draft)
builder.add_node("review", apply_feedback)
builder.add_node("finalize", finalize_content)
builder.add_node("escalate", escalate_content)

builder.set_entry_point("write")
builder.add_edge("write", "review")
builder.add_conditional_edges(
    "review",
    route_after_feedback,
    {
        "revise": "write",
        "finalize": "finalize",
        "escalate": "escalate"
    }
)
builder.add_edge("finalize", END)
builder.add_edge("escalate", END)

graph = builder.compile(
    checkpointer=MemorySaver(),
    interrupt_before=["review"]   # pause at every review gate
)

config = {"configurable": {"thread_id": "revision-loop-001"}}

# Initial run
graph.invoke(
    {"request": "Blog post about AI safety", "feedback_history": [], "accepted": False, "iteration": 0},
    config
)

# Feedback loop
while True:
    snapshot = graph.get_state(config)
    if not snapshot.next:
        print("\nGraph complete.")
        print(f"Final: {snapshot.values['final']}")
        break

    current_draft = snapshot.values["draft"]
    print(f"\nCurrent draft: {current_draft}")
    
    accept = input("Accept? (yes/no): ").strip().lower()
    
    if accept == "yes":
        graph.update_state(config, {
            "accepted": True,
            "feedback_history": [{"iteration": snapshot.values["iteration"], "comment": "Accepted"}]
        })
    else:
        comment = input("Feedback: ").strip()
        graph.update_state(config, {
            "accepted": False,
            "feedback_history": [{"iteration": snapshot.values["iteration"], "comment": comment}]
        })
    
    graph.invoke(None, config)

In [ ]:
## 6.2 Pattern: Confidence-Gated Feedback
# Use HITL only when the agent is uncertain. If confidence is high, proceed automatically; if low, 
# interrupt for human input.


class ConfidenceState(TypedDict):
    query: str
    answer: str
    confidence: float   # 0.0 - 1.0
    human_verified: bool
    final_answer: str

CONFIDENCE_THRESHOLD = 0.8

def answer_node(state: ConfidenceState) -> dict:
    """Agent answers and self-reports confidence."""
    # Simulate: in real code this comes from LLM structured output
    answer = f"Answer to '{state['query']}': 42"
    confidence = 0.65   # below threshold
    print(f"[answer_node] Confidence: {confidence}")
    return {"answer": answer, "confidence": confidence}

def should_interrupt(state: ConfidenceState) -> str:
    if state["confidence"] >= CONFIDENCE_THRESHOLD:
        return "auto_approve"
    return "needs_review"

def auto_approve_node(state: ConfidenceState) -> dict:
    print("[auto_approve] High confidence — proceeding automatically.")
    return {"final_answer": state["answer"], "human_verified": False}

def human_review_node(state: ConfidenceState) -> dict:
    """Interrupted here — human verifies."""
    return {}

def finalize_with_human(state: ConfidenceState) -> dict:
    return {"final_answer": state["answer"], "human_verified": True}

builder = StateGraph(ConfidenceState)
builder.add_node("answer", answer_node)
builder.add_node("auto_approve", auto_approve_node)
builder.add_node("human_review", human_review_node)
builder.add_node("finalize", finalize_with_human)

builder.set_entry_point("answer")
builder.add_conditional_edges(
    "answer",
    should_interrupt,
    {"auto_approve": "auto_approve", "needs_review": "human_review"}
)
builder.add_edge("auto_approve", END)
builder.add_edge("human_review", "finalize")
builder.add_edge("finalize", END)

graph = builder.compile(
    checkpointer=MemorySaver(),
    interrupt_before=["human_review"]
)

config = {"configurable": {"thread_id": "confidence-001"}}
result = graph.invoke({"query": "What is the capital of Jupiter?"}, config)

snapshot = graph.get_state(config)
if snapshot.next:
    print(f"\nLow confidence ({result['confidence']}). Human review needed.")
    print(f"Proposed answer: {result['answer']}")
    
    corrected = input("Correct answer (or press Enter to accept): ").strip()
    if corrected:
        graph.update_state(config, {"answer": corrected})
    
    final = graph.invoke(None, config)
    print(f"\nFinal: {final['final_answer']} (human_verified: {final['human_verified']})")
else:
    print(f"\nAuto-approved: {result['final_answer']}")

#### Part 7 — Time Travel (Revisiting Past Checkpoints)

LangGraph saves every checkpoint as the graph runs. You can list all saved checkpoints and re-run from any past point — effectively "undoing" agent actions.

``` python
# List all checkpoints for a thread
for checkpoint_tuple in graph.get_state_history(config):
    print(f"Step: {checkpoint_tuple.metadata.get('step')}")
    print(f"State: {checkpoint_tuple.values}")
    print(f"Next: {checkpoint_tuple.next}")
    print("---")
```

To re-run from a specific checkpoint:

``` python 
# Get all checkpoints
history = list(graph.get_state_history(config))

# Go back to step 2
target_checkpoint = history[-3]   # history is newest-first

# Re-run from that checkpoint's config
replay_config = target_checkpoint.config
result = graph.invoke(None, replay_config)
```

This is powerful for: recovering from bad agent actions, testing "what if the human had approved differently", and debugging multi-step failures.



#### Part 8 — Production Patterns and Best Practices

* 8.1 Always Use a Thread ID Per Conversation

``` python
import uuid

def start_workflow(user_id: str, task: str):
    thread_id = f"{user_id}-{uuid.uuid4()}"
    config = {"configurable": {"thread_id": thread_id}}
    return config
```

* 8.2 Expose State Cleanly to Human Reviewers

``Build a get_review_payload() helper that extracts only what the reviewer needs:``

``` python
def get_review_payload(graph, config) -> dict:
    snapshot = graph.get_state(config)
    return {
        "is_paused": bool(snapshot.next),
        "waiting_on": list(snapshot.next),
        "current_state": snapshot.values,
        "iteration": snapshot.metadata.get("step", 0)
    }
```

* 8.3 Timeout Handling

``Interrupted graphs wait forever by default. In production, set a timeout policy:``

``` python 
import time

def wait_for_human_with_timeout(graph, config, timeout_seconds=3600):
    start = time.time()
    while True:
        snapshot = graph.get_state(config)
        if not snapshot.next:
            return "completed"
        if time.time() - start > timeout_seconds:
            # Auto-reject or escalate
            graph.update_state(config, {"human_decision": "timeout_rejected"})
            graph.invoke(None, config)
            return "timeout"
        time.sleep(30)  # poll every 30 seconds
```

* 8.4 Audit Trail via Feedback History

`` Always accumulate feedback into state. Every human interaction should leave a trace: ``

``` python
class AuditableState(TypedDict):
    content: str
    audit_log: Annotated[List[dict], operator.add]

# When human takes action:
graph.update_state(config, {
    "audit_log": [{
        "timestamp": datetime.now().isoformat(),
        "reviewer": "alice@company.com",
        "action": "approved",
        "notes": "Reviewed and confirmed accuracy"
    }]
})
```

* 8.5 Distinguishing Interrupt Types

``` python
def classify_interrupt(snapshot) -> str:
    """Classify why the graph is paused."""
    if not snapshot.next:
        return "completed"
    
    next_node = snapshot.next[0]
    
    if "approve" in next_node or "review" in next_node:
        return "awaiting_approval"
    elif "tools" in next_node:
        return "awaiting_tool_approval"
    elif "feedback" in next_node:
        return "awaiting_feedback"
    else:
        return "awaiting_human_input"
```

